# 1. What you'll learn

        - turn a panel into leakage-safe lag rows
- compare global LightGBM with per-series ARIMA
- show negative transfer when series relationships change

        **The one question this notebook answers:** When does one lag-feature model shared across series beat a separate ARIMA per series?

# 2. Setup

This lesson keeps global forecasting across many series small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from lightgbm import LGBMRegressor
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Eighteen related monthly demand series share weather-like seasonality and trend but have different levels and amplitudes. The last six months of every series are held out.

**Small example:** For one series [10,12,11,15,16], a row at time 5 can use lags 16 and 15 plus earlier rolling means—but never the target at time 5 or later.

In [ ]:
rng=np.random.default_rng(SEED);S=18;T=72;rows=[]
for s in range(S):
 level=25+rng.uniform(0,30);amp=rng.uniform(3,9);trend=rng.uniform(.03,.18);phase=rng.normal(0,.12)
 noise=rng.normal(0,1.8,T);values=level+trend*np.arange(T)+amp*np.sin(2*np.pi*(np.arange(T)/12+phase))+noise
 for t,v in enumerate(values): rows.append((s,t,v))
frame=pd.DataFrame(rows,columns=["series","time","demand"]);print("Panel:",frame.shape,"series:",frame.series.nunique(),"periods/series:",T);print(frame.groupby("series").demand.mean().head().round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
pivot=frame.pivot(index="time",columns="series",values="demand");fig,axes=plt.subplots(1,3,figsize=(14,3.7));pivot.iloc[:,:5].plot(ax=axes[0],legend=False);axes[0].set_title("Five related series")
pivot.mean().hist(ax=axes[1],bins=12,color="#176b66");axes[1].set_title("Different series levels")
pivot.diff(12).stack().hist(ax=axes[2],bins=25,color="#d38b45");axes[2].set_title("Annual differences");plt.tight_layout();plt.show()

**Takeaway:** Series share a seasonal rhythm that a global model can pool.

**Takeaway:** Different levels require per-series history features rather than one common intercept.

**Takeaway:** Annual differences are smaller than raw levels, validating lag-12 as a strong baseline.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
def features(df):
 out=df.sort_values(["series","time"]).copy();g=out.groupby("series").demand;out["lag1"]=g.shift(1);out["lag2"]=g.shift(2);out["lag12"]=g.shift(12);out["roll3"]=g.shift(1).rolling(3).mean().reset_index(level=0,drop=True);out["month"]=out.time%12;return out.dropna()
tab=features(frame);train=tab[tab.time<T-6];test=tab[tab.time>=T-6];cols=["series","time","month","lag1","lag2","lag12","roll3"];print("Supervised train/test:",train.shape,test.shape,"latest train time:",train.time.max())

# 6. Train

        Training turns the assumptions behind global forecasting across many series into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Create each row using only earlier lags and rolling values.
2. Pool all series into one supervised table.
3. Fit shallow boosted trees that share lag relationships.
4. Forecast held-out rows and compare with separate univariate models.

In [ ]:
global_model=LGBMRegressor(n_estimators=180,num_leaves=15,learning_rate=.04,min_child_samples=20,verbosity=-1,random_state=SEED,n_jobs=1).fit(train[cols],train.demand);gp=global_model.predict(test[cols])
arima_pred=[]
for s in range(S):
 hist=pivot[s].iloc[:-6];arima_pred.extend(ARIMA(hist,order=(1,1,1)).fit().forecast(6))
arima_pred=np.array(arima_pred);seasonal=test.lag12.to_numpy();print("Prediction arrays:",gp.shape,arima_pred.shape)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
scores={"seasonal naive":mean_absolute_error(test.demand,seasonal),"per-series ARIMA":mean_absolute_error(test.demand,arima_pred),"global LightGBM":mean_absolute_error(test.demand,gp)};per=pd.DataFrame({"series":test.series,"truth":test.demand,"global":gp,"arima":arima_pred,"naive":seasonal}).groupby("series").apply(lambda z:pd.Series({"global":mean_absolute_error(z.truth,z["global"]),"arima":mean_absolute_error(z.truth,z.arima),"naive":mean_absolute_error(z.truth,z.naive)}),include_groups=False)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(scores.keys(),scores.values(),color=[".6","#d38b45","#176b66"]);axes[0].tick_params(axis="x",rotation=20);axes[0].set_title("Panel MAE")
axes[1].scatter(per.arima,per["global"]);m=max(per.max());axes[1].plot([0,m],[0,m],"--",color="black");axes[1].set(title="Per-series errors",xlabel="ARIMA MAE",ylabel="global MAE")
z=test[test.series==0];axes[2].plot(z.time,z.demand,marker="o",label="truth");axes[2].plot(z.time,gp[test.series.to_numpy()==0],marker="o",label="global");axes[2].plot(z.time,seasonal[test.series.to_numpy()==0],marker="o",label="naive");axes[2].legend();axes[2].set_title("One held-out series");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print({k:round(v,3) for k,v in scores.items()})

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Global pooling causes negative transfer when a series follows a relationship absent from the training population. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
bad=frame.copy();mask=bad.series==0;bad.loc[mask,"demand"]=bad.loc[mask,"demand"].mean()-(bad.loc[mask,"demand"]-bad.loc[mask,"demand"].mean());badtab=features(bad);badtest=badtab[(badtab.series==0)&(badtab.time>=T-6)];bp=global_model.predict(badtest[cols])
print(f"Global MAE on changed-relationship series={mean_absolute_error(badtest.demand,bp):.3f}; seasonal naive={mean_absolute_error(badtest.demand,badtest.lag12):.3f}")

**Use this when…** many related series share lag/seasonal behaviour and individual histories are short.

        **Don't use this when…** series are unrelated, future-known features are unavailable, or recursive horizons amplify lag errors unchecked.

        ## Try this

        1. Remove series ID and measure whether levels are still recoverable.
2. Hold out an entire series to test zero-shot transfer.
3. Forecast recursively for 18 months and watch error accumulation.